<a href="https://www.kaggle.com/code/mumerfarooqbajwa/cleaning-messy-clinic-appointments-dataset?scriptVersionId=338085141" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nudratabbas/messy-clinic-appointments-dataset/messy_clinic_appointments.csv


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nudratabbas/messy-clinic-appointments-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/nudratabbas/messy-clinic-appointments-dataset


In [3]:
import pandas as pd
import numpy as np

from sklearn import set_config
set_config(transform_output='pandas')

In [4]:
df = pd.read_csv("/kaggle/input/datasets/nudratabbas/messy-clinic-appointments-dataset/messy_clinic_appointments.csv")
df.sample(10)

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
266,1088,Amy King,49,Female,15-Sep-2025,2025/01/30,Peter Wells,General,£241.48,Y
990,1052,Nicholas Mcdaniel,48,F,"August 09, 25",2025/04/26,Daniel Smith,Orthopedics,€79.73,0
55,1068,Vanessa Jones,63,F,2025/11/09,10-Sep-2025,Courtney Knight,Orthopedics,$169.25,Yes
394,1005,Mr. Phillip Baxter,80,M,07/24/2025,01/16/2025,Tricia Hunt,General,$320.72,N
257,1056,Joshua Sanford,57,female,2025/06/20,13-Feb-2025,Tara Montoya,Neurology,$186.61,1
245,1060,Haley Taylor,51,female,2025/07/06,04/18/2025,Christopher Wolfe,General,Rs433.55,Y
584,1069,Jacqueline Peters,40,0,2025/12/09,11-Oct-2025,Adam Jensen,General,€266.74,Yes
934,1091,Scott Walker,62,Male,03/22/2026,2025/11/28,Brianna Hernandez,Neurology,$433.63,Y
401,1074,Michelle Edwards,30,female,03-Dec-2025,07/27/2025,Charles Harvey,Orthopedics,£242.2,Yes
757,1018,Joel Marshall,55,1,12/23/2025,23-Feb-2025,Victoria Garcia,Neurology,€306.8,0


# **Preprocessing/Cleaning of Messy Data**

In [5]:
df['gender'].unique()

array(['female', 'F', 'M', nan, '1', 'Female', 'male', 'Male', '0'],
      dtype=object)

In [6]:
df['gender'].str.lower()

0      female
1           f
2           m
3           f
4         NaN
        ...  
995    female
996         f
997         1
998         f
999      male
Name: gender, Length: 1000, dtype: object

In [7]:
df['gender']= np.where(
    (df['gender'].str.lower() == 'female' ) | (df['gender']=='F') |(df['gender']=='0') ,
    'F',
    np.where(
        (df['gender'].str.lower() == 'male') | (df['gender']=='M') |(df['gender']=='1') ,
            'M',
        df['gender']
    )
)

In [8]:
df['gender'].unique()

array(['F', 'M', nan], dtype=object)

# Change the 'appointment_date' Column to Same Date Format 

In [9]:
df['appointment_date']

0            2026/02/26
1            05/23/2025
2           30-Nov-2025
3            May 18, 25
4            2026/03/07
             ...       
995    September 14, 25
996        April 09, 25
997     February 13, 26
998          2025/07/04
999         05-Nov-2025
Name: appointment_date, Length: 1000, dtype: object

In [10]:
df['appointment_date']= pd.to_datetime(df['appointment_date'],format='mixed')

In [11]:
df['appointment_date'].isnull().sum()

np.int64(0)

# Change 'booking_date' Column to Same Format of DateTime

In [12]:
df['booking_date']

0         2024/12/03
1        12-Jun-2024
2      August 05, 24
3         09/09/2024
4         08/17/2024
           ...      
995       2024/07/15
996      June 20, 24
997       2024/06/13
998       2024/04/15
999       2024/07/22
Name: booking_date, Length: 1000, dtype: object

In [13]:
df['booking_date'] = pd.to_datetime(df['booking_date'], format='mixed'	)

In [14]:
df['booking_date']

0     2024-12-03
1     2024-06-12
2     2024-08-05
3     2024-09-09
4     2024-08-17
         ...    
995   2024-07-15
996   2024-06-20
997   2024-06-13
998   2024-04-15
999   2024-07-22
Name: booking_date, Length: 1000, dtype: datetime64[ns]

In [15]:
df['booking_date'].isnull().sum()

np.int64(0)

# Splitting 'billing_amount' Column to Get billing_amount in Same Currency

In [16]:
df['billing_amount'].unique()

array(['£425.8', '€344.26', '€203.34', 'Rs85.76', '$84.44', nan, '€99.0',
       'Rs374.63', '$452.37', '€494.97', '£91.91', '$82.88', 'Rs277.14',
       '€411.01', '£67.51', '€163.99', '£252.38', '$153.07', 'Rs329.3',
       '€135.45', 'Rs72.73', 'Rs258.64', '$203.61', 'Rs236.22',
       'Rs292.23', '$51.72', '£218.84', 'Rs160.38', 'Rs342.38',
       'Rs208.91', 'Rs426.86', '£215.31', '£413.53', '£396.22', '$340.25',
       '$481.44', '$442.77', 'Rs175.63', '€320.66', '$268.14', '€499.75',
       'Rs452.11', 'Rs479.77', '£213.35', 'Rs120.34', 'Rs135.43',
       '€214.44', 'Rs376.08', '€343.51', 'Rs185.66', '€200.01', '$367.06',
       'Rs393.4', '£387.4', '$169.25', '$274.81', 'Rs68.81', 'Rs365.13',
       '$466.97', 'Rs484.52', '€114.45', '€195.21', '$149.8', 'Rs270.33',
       '$186.24', 'Rs425.63', '£91.77', '€91.27', '£453.06', '€61.37',
       '$185.52', '£112.48', '$382.34', 'Rs168.18', '$206.53', '$258.21',
       '€253.4', 'Rs83.09', '$80.38', 'Rs260.4', '$294.58', '$247.13',


In [17]:
df[ ((df['billing_amount'].str[0]!='£') &  (df['billing_amount'].str[0] != '€') & (df['billing_amount'].str[0] != '$') & (df['billing_amount'].str[0] != "R"))] 
# only four currencies
# £, €, Rs, $


,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,billing_amount,follow_up_required
5,1057,Joe Sharp,78,M,2025-09-26,2025-08-31,Kimberly Fields,General,NaN,Yes
33,1041,Nicole Shaw,20,M,2025-05-07,2025-01-19,Mr. Brian Morton,Neurology,NaN,Y
75,1061,Jessica Smith,66,F,2025-08-20,2025-03-08,Crystal Mack,Neurology,NaN,Yes
97,1052,Keith Becker,60,F,2025-11-07,2024-08-28,Kristopher Collier,General,NaN,Y
140,1069,Thomas Landry,60,M,2025-09-03,2025-08-01,Brandon Moore,General,NaN,No
145,1047,John Campbell,67,F,2026-02-10,2024-06-21,Tanya Baxter,Orthopedics,NaN,N
159,1038,Lindsay Rodriguez,39,M,2025-10-23,2025-07-18,Troy Lynch,Orthopedics,NaN,0
176,1091,David Martinez,80,M,2025-09-13,2025-07-07,Amber Braun,Cardiology,NaN,Y
195,1087,Kimberly Guzman,31,F,2025-11-18,2025-04-02,Kristin Reeves,General,NaN,Y
196,1056,Terry Miller,24,F,2025-04-22,2025-03-24,John Lyons,General,NaN,Yes


In [18]:
currencies_symbols =['£', '€', 'Rs', '$']

for symbol in currencies_symbols:
    length = len(symbol)
    df ['billing_amount_'+symbol] = np.where(df['billing_amount'].str[0:length]==symbol ,
                                  (df['billing_amount'].str[length:]) ,
                                   0)

In [19]:
df[['billing_amount','billing_amount_£','billing_amount_$','billing_amount_Rs','billing_amount_€']]

,billing_amount,billing_amount_£,billing_amount_$,billing_amount_Rs,billing_amount_€
0,£425.8,425.8,0,0,0
1,€344.26,0,0,0,344.26
2,€203.34,0,0,0,203.34
3,Rs85.76,0,0,85.76,0
4,$84.44,0,84.44,0,0
...,...,...,...,...,...
995,Rs200.43,0,0,200.43,0
996,Rs244.49,0,0,244.49,0
997,£154.48,154.48,0,0,0
998,$434.95,0,434.95,0,0


In [20]:
df['billing_amount'].unique()

array(['£425.8', '€344.26', '€203.34', 'Rs85.76', '$84.44', nan, '€99.0',
       'Rs374.63', '$452.37', '€494.97', '£91.91', '$82.88', 'Rs277.14',
       '€411.01', '£67.51', '€163.99', '£252.38', '$153.07', 'Rs329.3',
       '€135.45', 'Rs72.73', 'Rs258.64', '$203.61', 'Rs236.22',
       'Rs292.23', '$51.72', '£218.84', 'Rs160.38', 'Rs342.38',
       'Rs208.91', 'Rs426.86', '£215.31', '£413.53', '£396.22', '$340.25',
       '$481.44', '$442.77', 'Rs175.63', '€320.66', '$268.14', '€499.75',
       'Rs452.11', 'Rs479.77', '£213.35', 'Rs120.34', 'Rs135.43',
       '€214.44', 'Rs376.08', '€343.51', 'Rs185.66', '€200.01', '$367.06',
       'Rs393.4', '£387.4', '$169.25', '$274.81', 'Rs68.81', 'Rs365.13',
       '$466.97', 'Rs484.52', '€114.45', '€195.21', '$149.8', 'Rs270.33',
       '$186.24', 'Rs425.63', '£91.77', '€91.27', '£453.06', '€61.37',
       '$185.52', '£112.48', '$382.34', 'Rs168.18', '$206.53', '$258.21',
       '€253.4', 'Rs83.09', '$80.38', 'Rs260.4', '$294.58', '$247.13',


In [21]:
df.drop(columns=['billing_amount'], inplace=True)

In [22]:
df

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,follow_up_required,billing_amount_£,billing_amount_€,billing_amount_Rs,billing_amount_$
0,1080,Tammy Williams,76,F,2026-02-26,2024-12-03,Christopher Graham,Cardiology,1,425.8,0,0,0
1,1074,Megan Strickland,69,F,2025-05-23,2024-06-12,Brandon Lewis,Orthopedics,Y,0,344.26,0,0
2,1067,Amanda Schroeder,79,M,2025-11-30,2024-08-05,Deanna Edwards,Neurology,Y,0,203.34,0,0
3,1072,Anthony Mcpherson,47,F,2025-05-18,2024-09-09,Karen Parsons,General,No,0,0,85.76,0
4,1092,Benjamin Brown,45,NaN,2026-03-07,2024-08-17,Andrea Hernandez,General,1,0,0,0,84.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1082,Thomas Roman,64,F,2025-09-14,2024-07-15,Robert Bass,General,Yes,0,0,200.43,0
996,1008,Jack Cooper,67,F,2025-04-09,2024-06-20,Christopher Smith,General,No,0,0,244.49,0
997,1022,Christopher Brown,28,M,2026-02-13,2024-06-13,Kimberly Johnson,Neurology,N,154.48,0,0,0
998,1088,Michael Sullivan,65,F,2025-07-04,2024-04-15,Anthony Mccoy,Neurology,0,0,0,0,434.95


# Cleaning follow_up_required Column

In [23]:
df['follow_up_required'].unique()

array(['1', 'Y', 'No', 'Yes', 'N', '0'], dtype=object)

In [24]:
df['follow_up_required']= np.where (
    ( (df['follow_up_required']=='1') | (df['follow_up_required']=='Y') | (df['follow_up_required']=='Yes')),
1,
0)

In [25]:
df['follow_up_required'].unique()

array([1, 0])

In [26]:
df

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,follow_up_required,billing_amount_£,billing_amount_€,billing_amount_Rs,billing_amount_$
0,1080,Tammy Williams,76,F,2026-02-26,2024-12-03,Christopher Graham,Cardiology,1,425.8,0,0,0
1,1074,Megan Strickland,69,F,2025-05-23,2024-06-12,Brandon Lewis,Orthopedics,1,0,344.26,0,0
2,1067,Amanda Schroeder,79,M,2025-11-30,2024-08-05,Deanna Edwards,Neurology,1,0,203.34,0,0
3,1072,Anthony Mcpherson,47,F,2025-05-18,2024-09-09,Karen Parsons,General,0,0,0,85.76,0
4,1092,Benjamin Brown,45,NaN,2026-03-07,2024-08-17,Andrea Hernandez,General,1,0,0,0,84.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1082,Thomas Roman,64,F,2025-09-14,2024-07-15,Robert Bass,General,1,0,0,200.43,0
996,1008,Jack Cooper,67,F,2025-04-09,2024-06-20,Christopher Smith,General,0,0,0,244.49,0
997,1022,Christopher Brown,28,M,2026-02-13,2024-06-13,Kimberly Johnson,Neurology,0,154.48,0,0,0
998,1088,Michael Sullivan,65,F,2025-07-04,2024-04-15,Anthony Mccoy,Neurology,0,0,0,0,434.95


# Splitting appointment_date and booking_date Columns

In [27]:
dates= ['booking_date', 'appointment_date']

for d in dates:
    df[d+'_year'] = df[d].dt.year
    df[d+'_month'] = df[d].dt.month
    df[d+'_day'] =df[d].dt.day

In [28]:
df

,patient_id,patient_name,age,gender,appointment_date,booking_date,doctor,department,follow_up_required,billing_amount_£,billing_amount_€,billing_amount_Rs,billing_amount_$,booking_date_year,booking_date_month,booking_date_day,appointment_date_year,appointment_date_month,appointment_date_day
0,1080,Tammy Williams,76,F,2026-02-26,2024-12-03,Christopher Graham,Cardiology,1,425.8,0,0,0,2024,12,3,2026,2,26
1,1074,Megan Strickland,69,F,2025-05-23,2024-06-12,Brandon Lewis,Orthopedics,1,0,344.26,0,0,2024,6,12,2025,5,23
2,1067,Amanda Schroeder,79,M,2025-11-30,2024-08-05,Deanna Edwards,Neurology,1,0,203.34,0,0,2024,8,5,2025,11,30
3,1072,Anthony Mcpherson,47,F,2025-05-18,2024-09-09,Karen Parsons,General,0,0,0,85.76,0,2024,9,9,2025,5,18
4,1092,Benjamin Brown,45,NaN,2026-03-07,2024-08-17,Andrea Hernandez,General,1,0,0,0,84.44,2024,8,17,2026,3,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1082,Thomas Roman,64,F,2025-09-14,2024-07-15,Robert Bass,General,1,0,0,200.43,0,2024,7,15,2025,9,14
996,1008,Jack Cooper,67,F,2025-04-09,2024-06-20,Christopher Smith,General,0,0,0,244.49,0,2024,6,20,2025,4,9
997,1022,Christopher Brown,28,M,2026-02-13,2024-06-13,Kimberly Johnson,Neurology,0,154.48,0,0,0,2024,6,13,2026,2,13
998,1088,Michael Sullivan,65,F,2025-07-04,2024-04-15,Anthony Mccoy,Neurology,0,0,0,0,434.95,2024,4,15,2025,7,4


In [29]:
df[['appointment_date','appointment_date_year','appointment_date_month','appointment_date_day']]
df.drop(columns=['appointment_date','booking_date' ], inplace=True)

In [30]:
df

,patient_id,patient_name,age,gender,doctor,department,follow_up_required,billing_amount_£,billing_amount_€,billing_amount_Rs,billing_amount_$,booking_date_year,booking_date_month,booking_date_day,appointment_date_year,appointment_date_month,appointment_date_day
0,1080,Tammy Williams,76,F,Christopher Graham,Cardiology,1,425.8,0,0,0,2024,12,3,2026,2,26
1,1074,Megan Strickland,69,F,Brandon Lewis,Orthopedics,1,0,344.26,0,0,2024,6,12,2025,5,23
2,1067,Amanda Schroeder,79,M,Deanna Edwards,Neurology,1,0,203.34,0,0,2024,8,5,2025,11,30
3,1072,Anthony Mcpherson,47,F,Karen Parsons,General,0,0,0,85.76,0,2024,9,9,2025,5,18
4,1092,Benjamin Brown,45,NaN,Andrea Hernandez,General,1,0,0,0,84.44,2024,8,17,2026,3,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1082,Thomas Roman,64,F,Robert Bass,General,1,0,0,200.43,0,2024,7,15,2025,9,14
996,1008,Jack Cooper,67,F,Christopher Smith,General,0,0,0,244.49,0,2024,6,20,2025,4,9
997,1022,Christopher Brown,28,M,Kimberly Johnson,Neurology,0,154.48,0,0,0,2024,6,13,2026,2,13
998,1088,Michael Sullivan,65,F,Anthony Mccoy,Neurology,0,0,0,0,434.95,2024,4,15,2025,7,4


# Preprocessing Other Columns

In [31]:
df['patient_id'].value_counts()

patient_id
1074    17
1047    17
1058    17
1015    17
1089    16
        ..
1096     4
1079     4
1023     4
1000     3
1007     3
Name: count, Length: 101, dtype: int64

In [32]:
df['doctor'].value_counts()

doctor
Nancy Hernandez       2
Nancy Wilson          2
Wesley Johnson        2
Kristen Lopez         2
James Aguilar         2
                     ..
Nathaniel Thompson    1
Natalie Turner        1
Kurt Cain             1
Laura Edwards         1
Alex Richards         1
Name: count, Length: 990, dtype: int64

In [33]:
df['department'].value_counts()

department
Neurology      273
Orthopedics    262
Cardiology     234
General        231
Name: count, dtype: int64

In [34]:
df.drop(columns=['patient_id','patient_name','doctor'], inplace=True)

In [35]:
df

,age,gender,department,follow_up_required,billing_amount_£,billing_amount_€,billing_amount_Rs,billing_amount_$,booking_date_year,booking_date_month,booking_date_day,appointment_date_year,appointment_date_month,appointment_date_day
0,76,F,Cardiology,1,425.8,0,0,0,2024,12,3,2026,2,26
1,69,F,Orthopedics,1,0,344.26,0,0,2024,6,12,2025,5,23
2,79,M,Neurology,1,0,203.34,0,0,2024,8,5,2025,11,30
3,47,F,General,0,0,0,85.76,0,2024,9,9,2025,5,18
4,45,NaN,General,1,0,0,0,84.44,2024,8,17,2026,3,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,64,F,General,1,0,0,200.43,0,2024,7,15,2025,9,14
996,67,F,General,0,0,0,244.49,0,2024,6,20,2025,4,9
997,28,M,Neurology,0,154.48,0,0,0,2024,6,13,2026,2,13
998,65,F,Neurology,0,0,0,0,434.95,2024,4,15,2025,7,4


In [36]:
df.describe()

,age,follow_up_required,booking_date_year,booking_date_month,booking_date_day,appointment_date_year,appointment_date_month,appointment_date_day
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,53.750000,0.514000,2024.496000,6.565000,15.536000,2025.240000,6.645000,15.994000
std,21.137604,0.500054,0.525605,3.230203,8.724151,0.427297,3.530643,8.830059
min,18.000000,0.000000,2024.000000,1.000000,1.000000,2025.000000,1.000000,1.000000
25%,34.000000,0.000000,2024.000000,4.000000,8.000000,2025.000000,3.000000,8.000000
50%,55.000000,1.000000,2024.000000,6.000000,15.000000,2025.000000,7.000000,16.000000
75%,71.000000,1.000000,2025.000000,9.000000,23.000000,2025.000000,10.000000,24.000000
max,90.000000,1.000000,2026.000000,12.000000,31.000000,2026.000000,12.000000,31.000000


# Converting Numeric String into Numbers

In [37]:
# ['billing_amount_£','billing_amount_€', 'billing_amount_Rs', 'billing_amount_$']

In [38]:
cols= np.array(['billing_amount_£','billing_amount_€', 'billing_amount_Rs', 'billing_amount_$'])

for col in cols:
    df[col] = pd.to_numeric( df[col] )

In [39]:
df[['billing_amount_£','billing_amount_€', 'billing_amount_Rs', 'billing_amount_$']].describe()

,billing_amount_£,billing_amount_€,billing_amount_Rs,billing_amount_$
count,1000.000000,1000.00000,1000.000000,1000.000000
mean,60.480940,64.76146,71.895490,65.173140
std,128.370313,133.17135,139.889644,132.353334
min,0.000000,0.00000,0.000000,0.000000
25%,0.000000,0.00000,0.000000,0.000000
50%,0.000000,0.00000,0.000000,0.000000
75%,0.000000,0.00000,58.677500,0.000000
max,499.670000,499.75000,499.720000,498.840000


In [40]:
df

,age,gender,department,follow_up_required,billing_amount_£,billing_amount_€,billing_amount_Rs,billing_amount_$,booking_date_year,booking_date_month,booking_date_day,appointment_date_year,appointment_date_month,appointment_date_day
0,76,F,Cardiology,1,425.80,0.00,0.00,0.00,2024,12,3,2026,2,26
1,69,F,Orthopedics,1,0.00,344.26,0.00,0.00,2024,6,12,2025,5,23
2,79,M,Neurology,1,0.00,203.34,0.00,0.00,2024,8,5,2025,11,30
3,47,F,General,0,0.00,0.00,85.76,0.00,2024,9,9,2025,5,18
4,45,NaN,General,1,0.00,0.00,0.00,84.44,2024,8,17,2026,3,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,64,F,General,1,0.00,0.00,200.43,0.00,2024,7,15,2025,9,14
996,67,F,General,0,0.00,0.00,244.49,0.00,2024,6,20,2025,4,9
997,28,M,Neurology,0,154.48,0.00,0.00,0.00,2024,6,13,2026,2,13
998,65,F,Neurology,0,0.00,0.00,0.00,434.95,2024,4,15,2025,7,4


# Imputing Missing Data in Gender

In [41]:
# imputing so that after encoding with one hot encoder can not produce gender_nan column.  

In [42]:
non_nan_gender= df[df['gender'].isnull() == False]


In [43]:
 df['gender'].sample(n=2)

241    F
60     F
Name: gender, dtype: object

In [44]:
count = df['gender'].isnull().sum()

random_vals= df[ df['gender'].isnull() == False]['gender'].sample(n=count)#, replace=True)

df.loc[df['gender'].isnull() , 'gender'] =  random_vals.values
df['gender'].unique()

array(['F', 'M'], dtype=object)

# Encoding Categorical data to Numeric

In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder 

ohe=  OneHotEncoder(drop='first')

ohe.set_output(transform="default")

trf= ColumnTransformer ([
    ('encoder',ohe, ['gender', 'department'])
],
    remainder = 'passthrough',
    verbose_feature_names_out=False
)


In [46]:
df=trf.fit_transform(df)

In [47]:
df

,gender_M,department_General,department_Neurology,department_Orthopedics,age,follow_up_required,billing_amount_£,billing_amount_€,billing_amount_Rs,billing_amount_$,booking_date_year,booking_date_month,booking_date_day,appointment_date_year,appointment_date_month,appointment_date_day
0,0.0,0.0,0.0,0.0,76.0,1.0,425.80,0.00,0.00,0.00,2024.0,12.0,3.0,2026.0,2.0,26.0
1,0.0,0.0,0.0,1.0,69.0,1.0,0.00,344.26,0.00,0.00,2024.0,6.0,12.0,2025.0,5.0,23.0
2,1.0,0.0,1.0,0.0,79.0,1.0,0.00,203.34,0.00,0.00,2024.0,8.0,5.0,2025.0,11.0,30.0
3,0.0,1.0,0.0,0.0,47.0,0.0,0.00,0.00,85.76,0.00,2024.0,9.0,9.0,2025.0,5.0,18.0
4,1.0,1.0,0.0,0.0,45.0,1.0,0.00,0.00,0.00,84.44,2024.0,8.0,17.0,2026.0,3.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.0,1.0,0.0,0.0,64.0,1.0,0.00,0.00,200.43,0.00,2024.0,7.0,15.0,2025.0,9.0,14.0
996,0.0,1.0,0.0,0.0,67.0,0.0,0.00,0.00,244.49,0.00,2024.0,6.0,20.0,2025.0,4.0,9.0
997,1.0,0.0,1.0,0.0,28.0,0.0,154.48,0.00,0.00,0.00,2024.0,6.0,13.0,2026.0,2.0,13.0
998,0.0,0.0,1.0,0.0,65.0,0.0,0.00,0.00,0.00,434.95,2024.0,4.0,15.0,2025.0,7.0,4.0


In [48]:
df.isnull().sum()

gender_M                  0
department_General        0
department_Neurology      0
department_Orthopedics    0
age                       0
follow_up_required        0
billing_amount_£          0
billing_amount_€          0
billing_amount_Rs         0
billing_amount_$          0
booking_date_year         0
booking_date_month        0
booking_date_day          0
appointment_date_year     0
appointment_date_month    0
appointment_date_day      0
dtype: int64

In [49]:
df

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split (df.drop(columns=['follow_up_required'], inplace=False ), df['follow_up_required'], test_size=0.25, random_state=42)


# Train with Logistic Regression

In [50]:
from sklearn.linear_model import LogisticRegression
reg = LogisticRegression (max_iter=3580)
reg.fit(X_train, y_train)

LogisticRegression(max_iter=3580)

In [51]:
predicted_y= reg.predict(X_test)

In [52]:
from sklearn.metrics import accuracy_score
accuracy_score(predicted_y , y_test)

0.464

# Train with Decision Tree

In [53]:
from sklearn.tree import DecisionTreeClassifier
dtree = DecisionTreeClassifier()
dtree.fit(X_train, y_train)

DecisionTreeClassifier()

In [54]:
predicted_y= dtree.predict(X_test)
accuracy_score (predicted_y, y_test)

0.528

***This dataset is just synthethic one and can not be used for appointment prediction, (because there is not relationship btw input features and target feature). So this dataset is just for practice of cleaning data / data engineering.***